# OASIS Alzheimer — 3-class training (Kaggle)

Subject-level, leak-free training skeleton for the 3-class task
(`no_dementia` / `very_mild` / `mild_moderate`). Mirrors `research.md`.

**Runtime:** Kaggle Notebook, GPU (T4/P100) ON. Dataset: `ninadaithal/imagesoasis`.

Pipeline: rebuild the subject-level split on Kaggle (same seed → same folds) →
central-slice filter → two-stage transfer learning → **subject-level** metrics.


## 1. Config

In [ ]:
import os, re, random, glob
from pathlib import Path
import numpy as np, pandas as pd

def resolve_data_root(preferred):
    """Return `preferred` if it has class folders; otherwise auto-find the OASIS
    `Data` dir anywhere under /kaggle/input (handles repo uploads that nest it
    under datasets/Data/, and differing dataset slugs)."""
    if Path(preferred).is_dir():
        return preferred
    for cand in glob.glob("/kaggle/input/**/Data", recursive=True):
        if (Path(cand) / "Non Demented").is_dir():
            return cand
    return preferred  # fall through -> scan() returns empty -> guard fires with a clear message

CFG = dict(
    # Public dataset default; auto-resolved below for repo uploads (e.g. topotisarkar/oasis-dataset).
    data_root   = resolve_data_root("/kaggle/input/imagesoasis/Data"),
    seed        = 42,
    test_frac   = 0.20,
    n_folds     = 5,
    run_folds   = [0],          # [0] = smoke-test one fold; set to [0,1,2,3,4] for full CV
    central_lo  = 128,          # keep slices 128..132 (narrow central band = ~5x less data, much faster).
    central_hi  = 132,          # widen back to 120..140 only if accuracy needs it.
    backbone    = "resnet18",   # "resnet18" (recommended baseline) | "resnet50" | "efficientnet_b0"
    img_size    = 224,
    batch_size  = 32,
    stage1_epochs = 3,          # frozen backbone, train head only
    stage2_epochs = 15,         # fine-tune backbone + head
    lr_head     = 1e-3,
    lr_backbone = 1e-5,
    weight_decay= 1e-4,
    patience    = 3,            # early stop on val macro-F1 (lower = fewer wasted epochs)
    num_workers = 2,
    use_sampler = True,         # WeightedRandomSampler for class balance (image-appropriate; see step 6).
    use_amp     = True,         # mixed precision -> ~2x faster on T4/P100 GPUs.
    cache_split = True,         # cache the scanned/split table so reruns skip the slow file scan.
)
CLASS_NAMES = ["no_dementia", "very_mild", "mild_moderate"]
print("data_root ->", CFG["data_root"])

def set_seed(s):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(CFG["seed"])

## 2. Rebuild the split on Kaggle

Deterministic and subject-grouped, so it reproduces the local folds exactly
(same seed). Moderate → Mild merge, central-slice filter applied here.


In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from PIL import Image

FOLDER_TO_LABEL = {           # raw folder -> 3-class label (Moderate folded into Mild)
    "Non Demented": 0, "Very mild Dementia": 1,
    "Mild Dementia": 2, "Moderate Dementia": 2,
}
SUBJECT_RE = re.compile(r"(OAS1_\d+)", re.IGNORECASE)
SLICE_RE   = re.compile(r"_(\d+)\.(?:jpg|jpeg|png)$", re.IGNORECASE)

def scan(data_root):
    rows = []
    for folder, label in FOLDER_TO_LABEL.items():
        d = Path(data_root) / folder
        for p in sorted(d.glob("*")):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
            sm, im = SUBJECT_RE.search(p.name), SLICE_RE.search(p.name)
            if not sm: continue
            rows.append(dict(filepath=str(p), subject_id=sm.group(1).upper(),
                             label=label, slice_idx=int(im.group(1)) if im else -1))
    return pd.DataFrame(rows)

def drop_unreadable(df):
    """Filter out images PIL can't decode (truncated/corrupt uploads). Reopens and
    fully loads each file, since verify() alone misses some truncation."""
    ok = []
    for p in df["filepath"]:
        try:
            with Image.open(p) as im:
                im.load()
            ok.append(True)
        except Exception:
            ok.append(False)
    ok = pd.Series(ok, index=df.index)
    bad = int((~ok).sum())
    if bad:
        print(f"WARNING: dropped {bad} unreadable image(s) of {len(df)} "
              f"({bad/len(df):.1%}). If this is most of the set, the Kaggle "
              f"upload is corrupt -> re-upload the dataset.")
    return df[ok].copy()

def assign_folds(slices, test_frac, n_folds, seed):
    subj = (slices.groupby("subject_id")["label"].first().reset_index()
            .sort_values("subject_id").reset_index(drop=True))
    y = subj["label"].to_numpy()
    dev_idx, _ = next(StratifiedShuffleSplit(1, test_size=test_frac,
                                             random_state=seed).split(subj, y))
    subj["fold"] = -1
    dev = subj.iloc[dev_idx].reset_index()
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
    for f, (_, val) in enumerate(skf.split(dev, dev["label"])):
        subj.loc[dev.loc[val, "index"].to_numpy(), "fold"] = f
    return slices.merge(subj[["subject_id", "fold"]], on="subject_id", how="left")

def build_central():
    """Scan files -> assign folds -> keep central slices -> drop unreadable ones."""
    slices = scan(CFG["data_root"])
    assert len(slices), (
        f"No images found under {CFG['data_root']!r}. Check the Add Input path / data_root. "
        f"Folders present under /kaggle/input: {glob.glob('/kaggle/input/*')}")
    slices = assign_folds(slices, CFG["test_frac"], CFG["n_folds"], CFG["seed"])
    central = slices[slices.slice_idx.between(CFG["central_lo"], CFG["central_hi"])].copy()
    central = drop_unreadable(central)          # verify only the slices we actually train on
    assert len(central), "All central-slice images were unreadable -> re-upload the dataset."
    return central

# Cache: scanning tens of thousands of files is slow, so save the split table and
# reuse it on later runs. The cache key includes the slice window, so changing
# central_lo/central_hi rebuilds automatically. Note: /kaggle/working starts empty
# each session unless you save the notebook output / enable persistence.
CACHE = Path(f"/kaggle/working/central_{CFG['central_lo']}_{CFG['central_hi']}.parquet")
if CFG["cache_split"] and CACHE.exists():
    central = pd.read_parquet(CACHE)
    print(f"loaded cached split from {CACHE.name}")
else:
    central = build_central()
    if CFG["cache_split"]:
        central.to_parquet(CACHE)
        print(f"saved split cache -> {CACHE.name}")

print(f"central {CFG['central_lo']}-{CFG['central_hi']} (readable): {len(central):,}")
print(central.groupby("fold")["subject_id"].nunique())

## 3. Dataset + transforms

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate end-truncated JPEGs

IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def build_tf(train):
    steps = [transforms.Resize((CFG["img_size"], CFG["img_size"]))]
    if train:
        steps += [transforms.RandomRotation(10),
                  transforms.RandomHorizontalFlip(0.5),
                  transforms.ColorJitter(brightness=0.1)]
    steps += [transforms.ToTensor(),
              transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(steps)

class OasisDS(Dataset):
    def __init__(self, df, train, retries=4):
        self.df = df.reset_index(drop=True); self.tf = build_tf(train)
        self.retries = retries

    def __len__(self): return len(self.df)

    def _load(self, i):
        r = self.df.iloc[i]
        img = Image.open(r.filepath).convert("L")           # grayscale
        img = Image.merge("RGB", (img, img, img))            # -> 3 channels
        return self.tf(img), int(r.label), r.subject_id

    def __getitem__(self, i):
        # Flaky GitHub-import mounts occasionally return partial reads; retry the
        # same file, then fall back to the next index so one bad read never
        # kills the epoch.
        for attempt in range(self.retries):
            try:
                return self._load((i + attempt) % len(self.df))
            except Exception as e:
                if attempt == self.retries - 1:
                    print(f"WARN: skipping unreadable {self.df.iloc[i].filepath} ({e})")
        return self._load((i + self.retries) % len(self.df))

## 4. Model factory (backbone-agnostic)

In [ ]:
import torch.nn as nn
from torchvision import models

def build_model(backbone, n_classes=3, dropout=0.4):
    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        feat = m.fc.in_features; head_attr = "fc"
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feat = m.fc.in_features; head_attr = "fc"
    elif backbone == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        feat = m.classifier[1].in_features; head_attr = "classifier"
    else:
        raise ValueError(backbone)
    head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat, n_classes))
    setattr(m, head_attr, head)
    return m

def set_backbone_trainable(model, backbone, trainable):
    head_attr = "classifier" if backbone == "efficientnet_b0" else "fc"
    for name, p in model.named_parameters():
        p.requires_grad = trainable or name.startswith(head_attr)

## 5. Train / eval helpers (subject-level metrics)

In [ ]:
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = CFG["use_amp"] and device == "cuda"   # mixed precision only helps (and is safe) on GPU

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total = 0.0
    with torch.set_grad_enabled(train):
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            if train: optimizer.zero_grad()
            # autocast runs the forward pass in fast float16 where it's safe.
            with torch.autocast(device_type="cuda", enabled=AMP):
                out = model(x); loss = criterion(out, y)
            if train:
                if scaler is not None:
                    # GradScaler prevents float16 gradients from underflowing to zero.
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            total += loss.item() * x.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def subject_eval(model, loader):
    model.eval()
    probs, subs, ys = [], [], []
    for x, y, s in loader:
        p = torch.softmax(model(x.to(device)), 1).cpu().numpy()
        probs.append(p); subs.extend(s); ys.extend(y.numpy())
    probs = np.concatenate(probs)
    df = pd.DataFrame(probs, columns=CLASS_NAMES); df["subject_id"] = subs; df["y"] = ys
    g = df.groupby("subject_id")
    pred = g[CLASS_NAMES].mean().values.argmax(1)      # mean-prob aggregation
    true = g["y"].first().values
    return dict(macro_f1=f1_score(true, pred, average="macro"),
                bal_acc=balanced_accuracy_score(true, pred),
                report=classification_report(true, pred, target_names=CLASS_NAMES, zero_division=0),
                cm=confusion_matrix(true, pred), true=true, pred=pred)

def class_weights(df):
    counts = df["label"].value_counts().sort_index().values
    w = counts.sum() / (len(counts) * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)

## 6. Two-stage training over the selected fold(s)

In [ ]:
from torch.utils.data import WeightedRandomSampler

def make_train_loader(tr):
    """Training loader. With use_sampler, draw rare-class images more often
    (WeightedRandomSampler) — the image-friendly way to fight class imbalance,
    instead of inventing fake images (SMOTE/ADASYN don't fit raw images)."""
    ds = OasisDS(tr, True)
    if CFG["use_sampler"]:
        cw = class_weights(tr).cpu().numpy()          # weight per class
        sample_w = cw[tr["label"].to_numpy()]         # weight per image
        sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
        return DataLoader(ds, CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True)
    return DataLoader(ds, CFG["batch_size"], shuffle=True,
                      num_workers=CFG["num_workers"], pin_memory=True)

def train_fold(fold):
    tr = central[(central.fold != fold) & (central.fold != -1)]
    va = central[central.fold == fold]
    print(f"\n=== fold {fold}: train {tr.subject_id.nunique()} subj / {len(tr)} img  "
          f"| val {va.subject_id.nunique()} subj / {len(va)} img ===")
    tr_dl = make_train_loader(tr)
    va_dl = DataLoader(OasisDS(va, False), CFG["batch_size"], shuffle=False,
                       num_workers=CFG["num_workers"], pin_memory=True)

    model = build_model(CFG["backbone"]).to(device)
    # If the sampler already balances classes, don't also weight the loss (would double-count).
    criterion = nn.CrossEntropyLoss() if CFG["use_sampler"] else nn.CrossEntropyLoss(weight=class_weights(tr))
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    # Stage 1 — frozen backbone, head only.
    set_backbone_trainable(model, CFG["backbone"], False)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=CFG["lr_head"], weight_decay=CFG["weight_decay"])
    for e in range(CFG["stage1_epochs"]):
        l = run_epoch(model, tr_dl, criterion, opt, scaler)
        print(f"  [s1 {e+1}] train_loss={l:.4f}")

    # Stage 2 — unfreeze, fine-tune with low LR; early-stop on val macro-F1.
    set_backbone_trainable(model, CFG["backbone"], True)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr_backbone"],
                            weight_decay=CFG["weight_decay"])
    best, best_state, wait = -1, None, 0
    for e in range(CFG["stage2_epochs"]):
        l = run_epoch(model, tr_dl, criterion, opt, scaler)
        m = subject_eval(model, va_dl)
        print(f"  [s2 {e+1}] train_loss={l:.4f}  val_macroF1={m['macro_f1']:.4f}  val_balAcc={m['bal_acc']:.4f}")
        if m["macro_f1"] > best:
            best, best_state, wait = m["macro_f1"], {k: v.cpu() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= CFG["patience"]: print("  early stop"); break
    model.load_state_dict(best_state)
    # Save the best weights so you can reload instead of retraining next session.
    ckpt = f"/kaggle/working/model_fold{fold}.pt"
    torch.save(best_state, ckpt)
    print(f"  saved -> {ckpt}  (best val_macroF1={best:.4f})")
    return model, subject_eval(model, va_dl)

results = {}
for f in CFG["run_folds"]:
    _, results[f] = train_fold(f)

## 7. Results (subject-level, vs 76.7% majority baseline)

In [ ]:
f1s  = [results[f]["macro_f1"] for f in results]
accs = [results[f]["bal_acc"]  for f in results]
print(f"macro-F1  : {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
print(f"bal-acc   : {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print("majority baseline accuracy = 0.767 (predict no_dementia for all)\n")
for f in results:
    print(f"--- fold {f} ---"); print(results[f]["report"])
    print("confusion matrix (rows=true, cols=pred):"); print(results[f]["cm"], "\n")

## Notes / next steps
- Set `run_folds = [0,1,2,3,4]` for full CV; report mean ± std (done above).
- Only evaluate the **test set** (`fold == -1`) once, after the recipe is frozen.
- Try `backbone = "resnet50"` as a second model and compare macro-F1.
- Add Grad-CAM on a few subjects for a sanity check against hidden leakage (research.md §9b).
